In [ ]:
import pandas as pd
import os
from pathlib import Path
import chardet
import re
print("current directory listing, locate your cve data file:")
print(f"=" * 70)
current_dir = Path('.')
for item in current_dir.iterdir():
    print(item.name)


### CAUSALITY Prediction Counter
Specify your CVE data in the next field, identify the field  containing CVEs which the fourth cell will try to identify for you.  The final output will be a count of predictions that exist in the local population. 

In [ ]:
vuln_path = 'kev.csv' # Identify the vuln data file to ingest

In [ ]:
with open(vuln_path, 'rb') as file:
    result = chardet.detect(file.read())
    encoding = result['encoding']
vulns = pd.read_csv(vuln_path, low_memory=False, encoding=encoding, header=0)
vulns.columns = vulns.columns.str.strip().str.lower()

all_nan = [col for col in vulns.columns if vulns[col].isna().all()]
if all_nan:
    print("\n⚠️ Fields entirely NaN:")
    for col in all_nan:
        print(f" - {col}")
else:
    print("\n✅ No fields are entirely NaN")

print("Shape of the vulns dataframe is:", vulns.shape)
print()
print("These are the fields in your dataframe:")
print(vulns.columns.tolist())

In [ ]:
# CVE pattern: CVE-YYYY-#####
cve_pattern = r'CVE-\d{4}-\d{4,7}'

# Search for the column containing CVEs
cve_column = None

for col in vulns.columns:
    try:
        # Convert to string and check for CVE pattern
        sample = vulns[col].dropna().head(50).astype(str)
        if sample.str.contains(cve_pattern, regex=True, na=False).any():
            cve_column = col
            break
    except:
        pass

if cve_column:
    print(f"✓ CVE field found: '{cve_column}'")
    print(f"\nFirst 5 values:")
    for i, cve in enumerate(vulns[cve_column].head(5), 1):
        print(f"  {i}. {cve}")
else:
    print("❌ No column with CVE pattern found")
    print("\nAvailable columns:")
    print(vulns.columns.tolist())
    print("\nSample data:")
    print(vulns.head())

In the output above, identify the your field name that contains CVEs and specify it in the cell below.

In [ ]:
# Check out the field list above and identify your field that contains CVE IDs. Provide it to the function below
# so that we have normalized field names across dataframes.

SOURCE_CVE_FIELD = 'cveid'  # <-- change this as needed
colmap = {c.lower(): c for c in vulns.columns}

if SOURCE_CVE_FIELD.lower() in colmap:
    src = colmap[SOURCE_CVE_FIELD.lower()]
    if src == 'cve':
        pass  # already named 'cve'
    elif 'cve' in vulns.columns:
        print("Target column 'cve' already exists; skipping rename to avoid duplicate.")
    else:
        vulns.rename(columns={src: 'cve'}, inplace=True)
else:
    print(f"Column '{SOURCE_CVE_FIELD}' not found; nothing to rename.")

In [ ]:
predictions = pd.read_csv('times.csv')
predictions.columns = predictions.columns.str.lower()
predictions['leadtimedays'] = predictions['leadtimedays'].astype(int)
print(f"Predictions loaded: {len(predictions)} rows")
print(f"Predictions columns: {predictions.columns.tolist()}")

In [ ]:
vulns_cves = vulns['cve'].unique()
# Filter predictions to only rows with CVEs that are in vulns
matching_predictions = predictions[predictions['cve'].isin(vulns_cves)]

print(f"This is the number we are interested in, you have: {len(matching_predictions)} prediction hits!")
print()
# Calculate statistics
leadtime = matching_predictions['leadtimedays'].dropna()
print(f"Lead Time Statistics (Matching Predictions)")
print(f"These numbers show how much early warning you would have had:")
print(f"=" * 50)
print(f"Count: {len(leadtime)}")
print(f"Mean: {leadtime.mean():.2f} days")
print(f"Median: {leadtime.median():.2f} days")
print(f"Std Dev: {leadtime.std():.2f} days")
print(f"Min: {leadtime.min():.0f} days")
print(f"Max: {leadtime.max():.0f} days")
print()
matching_predictions